# Customer Segmentation

### Finding hidden patterns in consumer behavior using unsupervised learning

**Table of Contents (to do)**

1. **Environmental Setup & Data Ingestion** — Environment configuration, library imports, and dataset loading.
2. **Data Diagnostics & Preprocessing** — Quality checks (missing values, duplicates) and statistical summaries.
3. **Exploratory Data Analysis (EDA)** — Visualizing distributions and spending trends across demographics.
4.  **Heuristic Segmentation:** Before applying complex algorithms, we attempt a logical, rule-based segmentation to establish a baseline understanding of our customer base (e.g., distinguishing "Savers" from "Spenders").
5. **K-Means Clustering Implementation** — Preprocessing, finding optimal $K$, and generating ML-based segments.
6. **Cluster Profiling & Interpretation** — Interpreting the characteristics of each cluster (e.g., "VIPs").
7. **Advanced Anomaly Detection (DBSCAN)** — Using DBSCAN to find noise and anomalies in the data.
8. **Strategic Insights & Conclusion** — Summarizing key findings, ranking segments by value, and exporting final results.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
Introduction and Project Overview</h1>

In the modern retail ecosystem, the "one-size-fits-all" marketing approach is obsolete. Success lies in precision—understanding not just *who* your customers are, but *how* they behave. This project leverages the **Mall Customers Dataset** to transform raw transactional data into actionable business intelligence through the power of **Unsupervised Machine Learning**.

Your analysis unfolds in a structured narrative:
1.  **Exploratory Data Analysis (EDA):** You begin by dissecting the demographic landscape—Age, Income, and Gender—to uncover initial correlations and hidden patterns.
2.  **Heuristic Segmentation:** Before applying complex algorithms, we attempt a logical, rule-based segmentation to establish a baseline understanding of our customer base (e.g., distinguishing "Savers" from "Spenders").
3.  **K-Means Clustering:** You implement the K-Means algorithm to mathematically group customers into distinct "groups". Please validate your cluster count ($K$) using rigorous techniques like the **Elbow Method** and **Silhouette Analysis**.
4.  **Advanced Anomaly Detection:** Finally, you deploy **DBSCAN** to identify outliers—unique customers who defy standard classification and may represent niche opportunities or data noise.

By the end of this notebook, you will have a segmented customer profile ready for targeted marketing strategies.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
1. Environmental Setup & Data Ingestion</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
1.1 Importing Libraries & Configuring Aesthetics</h2>

To ensure a robust analysis, you establish a foundational environment using standard Data Science libraries: **Pandas** and **NumPy** for data manipulation, and **Seaborn/Matplotlib** for visualization.

Crucially, we defined a custom **"Teal Corporate Palette"** at the start for you. This ensures that every chart generated in this report maintains a consistent, professional visual identity, avoiding the default rainbow colors that can distract from the insights.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
warnings.filterwarnings('ignore')

# Visual Aesthetics (The "Corporate Teal" Theme)
teal_corporate_palette = [
    "#5F9EA0", "#3A6F73", "#8FBFC1", 
    "#4A5D73", "#2C3E50", "#AAB7B8"
]
my_teal_color = "#3A6F73"
sns.set_palette(teal_corporate_palette)

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
1.2 Data Loading & Initial Inspection</h2>

You proceed by loading the `Mall_Customers.csv` dataset. The initial inspection involves checking the dataset's **dimensions (Shape)** to understand the volume of data you are working with, followed by peeking at the **Head** (first 5 rows) and **Tail** (last 5 rows). This step is vital to verify that the data has been ingested correctly and to get a first glimpse of the feature columns.

In [ ]:
path= '/kaggle/input/mall-customers/Mall_Customers.csv'
df= pd.read_csv(path)

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
2. Data Diagnostics & Preprocessing</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
2.1 Data Quality Checks</h2>

Before diving into analysis, we must ensure data integrity. We perform a "sanity check" to scan for **missing values (NaNs)** that could crash our models and **duplicate entries** that could skew our statistical results. We also use `.info()` to verify data types (integers vs. strings).

In [ ]:
df.info()

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
2.2 Statistical Summary</h2>

Understanding the "shape" of our variables is crucial. We separate our analysis into:
* **Numerical Statistics:** To check the central tendency (mean) and spread (std) of Age, Income, and Spending Score.
* **Categorical Statistics:** To see the frequency of non-numeric variables like Gender.

In [ ]:
# Statistical Summary
print("Numerical Statistics:")
display(df.describe().T)

print("\nCategorical Statistics:")
display(df.describe(include='object').T)

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
2.3 Feature Engineering & Cleaning</h2>

Raw data is rarely ready for modeling. In this step, we refine our dataset:
1.  **Drop `CustomerID`:** This is a unique identifier with no analytical value; keeping it would confuse our clustering algorithm.
2.  **Rename `Genre`:** We standardize the column name to `Gender` for clarity.
3.  **Binning (Feature Construction):** We create new categorical "buckets" for **Age** (e.g., Young, Senior) and **Income** (e.g., Low, High). This simplifies complex continuous data into interpretable groups for our initial analysis.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
3. Exploratory Data Analysis (EDA)</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
3.1 Univariate Analysis: Who are our customers?</h2>

We start by examining the distribution of our individual features. The count plots below reveal the composition of our dataset in terms of **Age Groups** and **Income Levels**. This gives us a baseline demographic profile.

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
3.2 Bivariate Analysis: What drives spending?</h2>

Next, we investigate how different demographics influence spending habits.
1.  **Boxplots:** We compare spending scores across Gender, Age, and Income. Surprisingly, we look for overlaps—if the boxes look similar, that feature might not be a strong segregator on its own.
2.  **Scatter Plot (The Golden Clusters):** This is the most critical visualization. By plotting **Income vs. Spending**, we can visually spot distinct groups (clusters) forming naturally. This confirms that segmentation is possible.

In [ ]:
# 2. The Golden Clusters Scatter Plot


<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
3.3 Multivariate Analysis: The "Spending Cliff"</h2>

We dive deeper to find complex patterns:
* **Pairplot:** To see pairwise relationships across all variables.
* **Violin Plot:** Combines boxplots with density estimation to show the "shape" of spending across ages.
* **Trend Line:** We calculate the average spending score for every 5-year age bin. This reveals a potential "Spending Cliff"—a specific age where customer spending drops significantly.

In [ ]:
# Trend Analysis


<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
3.4 Correlation Matrix & Heatmaps</h2>

Finally, we quantify the relationships using a **Correlation Matrix**. We check if Age, Income, or Gender have a strong linear relationship with Spending Score. We also use a Pivot Table heatmap to visualize the density of our customer base across Age and Income brackets.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
4. Heuristic Segmentation (Manual Logic)</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
4.1 VIP Customer Profile (Score > 80)</h2>

Before building complex models, we isolate the "Whales"—customers with a Spending Score above 80. Analyzing this elite group reveals key insights about our most valuable shoppers (e.g., are they mostly young? Male or Female?).

In [ ]:
top_spenders = df[df['Spending Score (1-100)'] > 80]

print(f"VIP Customer Analysis (Score > 80)")
print(f"Number of VIPs: {len(top_spenders)}")

In [ ]:
print("\n[ Demographics ]")
display(top_spenders[['Age', 'Annual Income (k$)']].describe().loc[['mean', 'min', 'max']].T)

print("\n[ Gender Distribution ]")
print(top_spenders['Gender'].value_counts(normalize=True).round(2))

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
4.2 The Quadrant Strategy (Median Split)</h2>

Here, we apply a classic business matrix. We calculate the **Median Income** and **Median Spending Score** to draw two crossing lines, dividing our customers into four distinct logical quadrants:

1.  **Target:** High Income, High Spend (The Ideal Customer).
2.  **Savers:** High Income, Low Spend (Potential to convert).
3.  **Careless:** Low Income, High Spend (Risk of churning).
4.  **Conservative:** Low Income, Low Spend.

The scatter plot below visualizes these 4 manually created segments.

In [17]:
income_median = df['Annual Income (k$)'].median()
score_median = df['Spending Score (1-100)'].median()

def classify_customer(row):
    if row['Annual Income (k$)'] > income_median and row['Spending Score (1-100)'] > score_median:
        return 'Target (High Income - High Spend)'
    elif row['Annual Income (k$)'] > income_median and row['Spending Score (1-100)'] <= score_median:
        return 'Savers (High Income - Low Spend)'
    elif row['Annual Income (k$)'] <= income_median and row['Spending Score (1-100)'] > score_median:
        return 'Careless (Low Income - High Spend)'
    else:
        return 'Conservative (Low Income - Low Spend)'

df['Customer_Category'] = df.apply(classify_customer, axis=1)

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)',
                hue='Customer_Category', s=100, edgecolor='white', linewidth=2)

plt.axvline(x=income_median, color='red', linestyle='--', alpha=0.5)
plt.axhline(y=score_median, color='red', linestyle='--', alpha=0.5)

plt.title('Manual Segmentation: The 4 Types of Customers')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\nCustomer Segments Distribution:")
display(df['Customer_Category'].value_counts())

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
5. K-Means Clustering Implementation</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
5.1 Preprocessing: Standardization</h2>

Machine Learning algorithms like K-Means calculate distance to determine similarity. If one feature has a large range (e.g., Income: 15,000–137,000) and another has a small range (e.g., Score: 1–100), the larger number will dominate the calculation.

To prevent this bias, we use **StandardScaler** to transform our data so that all features contribute equally to the result.

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
5.2 Determining Optimal Clusters (K)</h2>

How do we know if we should have 3, 4, or 5 customer segments? We don't guess; we use math.
1.  **Elbow Method:** We plot the error rate as we add more clusters. We look for the "elbow" point where adding more clusters stops giving us significant gains.
2.  **Silhouette Analysis:** This visualizes how well-separated the clusters are. A higher score means better-defined groups.

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
5.3 Model Fitting & Result Visualization</h2>

Based on the Elbow Method (which suggests K=5), we initialize our final K-Means model with **5 clusters**. The scatter plot below shows the mathematical reality of our customer segments, with the **Last X marks** indicating the centroid (center of gravity) for each group.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
6. Cluster Profiling & Interpretation</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
6.1 Statistical Profiling: Who are they?</h2>

Now that we have our clusters, we must interpret them. We calculate the **mean values** for Age, Income, and Spending Score for each cluster.

The Boxplots below act as a "fingerprint" for each group:
* **Income Boxplot:** Tells us if the group is wealthy or budget-conscious.
* **Spending Boxplot:** Tells us if they are big spenders or savers.
* **Age Boxplot:** Reveals if they are younger trends-setters or older established shoppers.

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
6.2 Business Labeling: Naming the Tribes</h2>

Cluster numbers (0, 1, 2, 3, 4) are meaningless to a marketing team. We need descriptive names.

We create a function `get_cluster_label` that looks at the centroids (average Income and Score) of each cluster and assigns a logical name:
* **VIP:** High Income & High Spend.
* **Saver:** High Income & Low Spend.
* **Careless:** Low Income & High Spend.
* **Conservative:** Low Income & Low Spend.
* **Average:** The middle ground.

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
7. Advanced Anomaly Detection (DBSCAN)</h1>

K-Means is excellent, but it has a flaw: it forces *every* customer into a cluster, even if they don't fit well.

To solve this, we introduce **DBSCAN** (Density-Based Spatial Clustering). Unlike K-Means, DBSCAN looks for high-density areas. If a point is in a low-density region (far from others), it labels it as **Noise (-1)**.

These "Outliers" are crucial. They might be:
1.  **Fraud cases.**
2.  **Unique high-value customers** who need special attention.
3.  **Data errors.**

<h1 style="
    background-color: #F4F6F7;
    color: #2C3E50;
    padding: 20px;
    text-align: center;
    border-radius: 10px;
    border-left: 8px solid #5F9EA0;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 6px rgba(0,0,0,0.1);
">
8. Strategic Insights & Conclusion</h1>

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
8.1 Which segment is most valuable?</h2>

Finally, we summarize our findings. The bar chart below ranks our new segments by **Average Spending Score**. This clearly identifies our "VIP" and "Careless" segments as the primary drivers of revenue, while "Savers" represent a huge opportunity for up-selling campaigns.

<h2 style="
    color: #3A6F73;
    font-size: 22px;
    margin-top: 30px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    border-bottom: 2px solid #E0E6E8;
    padding-bottom: 8px;
">
8.2 Deployment: Exporting the Results</h2>

In [ ]:
df_final = pd.read_csv(path)
df_final['Cluster'] = df['Cluster']
df_final['Cluster_Name'] = df['Cluster_Name']
df_final.to_csv('Mall_Customers_Segmented.csv', index=False)
display(df_final[['CustomerID', 'Cluster', 'Cluster_Name']].head())